In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
dim_employee = pd.read_excel("dim_employee.xlsx")
fact_workload = pd.read_excel("fact_workload.xlsx")
fact_record = pd.read_excel("fact_record.xlsx")

## Data Analysis - Answering primary business questions

#### 1.1: Analyze salary trends over tenure and department

In [17]:
merge_df = pd.merge(dim_employee,fact_record, on = "Employee_ID" , how = 'inner')
avg_dept_salary_trends = merge_df.groupby('Department')['Monthly_Salary'].mean().sort_values(ascending = False).round(2)
desired_output = pd.concat([avg_dept_salary_trends.head(3),avg_dept_salary_trends.tail(3)])
desired_output

Department
Legal               144473.37
Operations          141030.16
Customer Support    140720.25
IT                  139781.87
Engineering         139174.97
Finance             138635.16
Name: Monthly_Salary, dtype: float64

#### 1.2: Identify most common education levels per role

In [7]:
most_common_edu = dim_employee.groupby('Job_Title')['Education_Level'].agg(pd.Series.mode)
most_common_edu

Job_Title
Analyst           Bachelor
Consultant        Bachelor
Developer         Bachelor
Director            Master
Engineer          Bachelor
Manager           Bachelor
Specialist        Bachelor
Sr. Manager         Master
Technician     High School
Name: Education_Level, dtype: object

#### 1.3: Determine the impact of remote work on performance 

In [8]:
merge_df_1 = pd.merge(fact_workload,fact_record, on = "Employee_ID" , how = 'inner')
remote_work_performance = merge_df_1.pivot('Remote_Work_Frequency')[['Employee_Satisfaction_Score','Performance_Score']].mean().round(2)
remote_work_performance

,Employee_Satisfaction_Score,Performance_Score
Remote_Work_Frequency,,
0,3.49,3.20
20,3.49,3.19
50,3.48,3.18
80,3.46,3.19
100,3.48,3.21


#### 1.4: Calculate the average Monthly Salary for each Gender within each Job title to check for compensation parity.

In [10]:
merge_df = pd.merge(dim_employee, fact_record, on = "Employee_ID", how = "inner")
gender_compensation = merge_df.pivot_table(index = "Department", columns = "Gender", values = "Monthly_Salary", aggfunc = 'mean').fillna(0).round(2)
gender_compensation

Gender,Female,Male
Department,,
Customer Support,141092.03,140422.20
Engineering,143053.74,135940.05
Finance,137917.07,139241.34
HR,139896.31,140994.05
IT,139182.26,140272.53
Legal,148996.55,141018.44
Marketing,137110.07,142016.08
Operations,142083.16,140176.41
Sales,138122.15,141214.02


#### 1.5: Analyze employee retention trends by age group.


In [8]:
merge_df = pd.merge(dim_employee,fact_record, on = "Employee_ID" , how = 'inner')
merge_df["Age_groups"] = pd.cut(x=merge_df["Age"],bins=[18,30,45,60],labels=["18-30", "31-45", "46-60"],right=True)
employee_retention_by_age =  merge_df.pivot_table(index = 'Age_groups', columns = 'Resigned', values='Employee_ID',aggfunc='count').fillna(0)
employee_retention_by_age['Employee Resignation %'] = employee_retention_by_age[True]*100/(employee_retention_by_age[True]+employee_retention_by_age[False])

employee_retention_by_age

C:\Users\Hp\AppData\Local\Temp\ipykernel_1852\383868592.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  employee_retention_by_age =  merge_df.pivot_table(index = 'Age_groups', columns = 'Resigned', values='Employee_ID',aggfunc='count').fillna(0)


Resigned,False,True,Employee Resignation %
Age_groups,,,
18-30,16473,12159,42.466471
31-45,25369,10223,28.722747
46-60,28726,4700,14.060911


#### 1.6 Department-wise education level distribution.

In [10]:
dept_edu_dist = dim_employee.pivot_table(index = 'Department', columns = 'Education_Level',values='Employee_ID',aggfunc='count').fillna(0)
dept_edu_dist

Education_Level,Bachelor,High School,Master,PhD
Department,,,,
Customer Support,8318,1593,4390,473
Engineering,11295,2117,5927,670
Finance,2795,534,1506,170
HR,2247,459,1161,160
IT,11261,2142,5841,652
Legal,1110,222,600,77
Marketing,2313,438,1120,114
Operations,8363,1616,4506,501
Sales,8723,1538,4566,482


#### 1.7 Year-wise hiring trends in the company.

In [11]:
fact_record['Hire_Date'] = pd.to_datetime(fact_record['Hire_Date'])

yearly_hiring = fact_record[['Employee_ID','Hire_Date']]
yearly_hiring['Year'] = yearly_hiring['Hire_Date'].dt.year
yearly_hiring_merge = pd.merge(dim_employee,yearly_hiring, on= 'Employee_ID')
yearly_hiring_pivot = yearly_hiring_merge.pivot_table(index = 'Year', columns = 'Department',values='Employee_ID',aggfunc='count').fillna(0)
yearly_hiring_pivot

C:\Users\Hp\AppData\Local\Temp\ipykernel_16864\3989218701.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yearly_hiring['Year'] = yearly_hiring['Hire_Date'].dt.year


Department,Customer Support,Engineering,Finance,HR,IT,Legal,Marketing,Operations,Sales
Year,,,,,,,,,
1989,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1990,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1991,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1992,0.0,3.0,0.0,0.0,4.0,0.0,0.0,0.0,1.0
1993,2.0,2.0,1.0,0.0,2.0,0.0,0.0,1.0,1.0
1994,1.0,1.0,0.0,1.0,6.0,0.0,1.0,1.0,3.0
1995,2.0,7.0,1.0,1.0,7.0,1.0,2.0,4.0,7.0
1996,6.0,6.0,2.0,3.0,14.0,1.0,3.0,4.0,8.0
1997,11.0,12.0,4.0,3.0,14.0,0.0,8.0,13.0,14.0


#### 1.8 Gender distribution across departments.

In [12]:
gender_dist = dim_employee.pivot_table(index = 'Department', columns = 'Gender',values='Employee_ID',aggfunc='count',margins = True).fillna(0)
gender_dist

Gender,Female,Male,All
Department,,,
Customer Support,6574,8200,14774
Engineering,9099,10910,20009
Finance,2291,2714,5005
HR,1842,2185,4027
IT,8954,10942,19896
Legal,870,1139,2009
Marketing,1808,2177,3985
Operations,6710,8276,14986
Sales,6852,8457,15309


#### 1.9 Department wise attrition rates.

In [13]:
merge_df = pd.merge(dim_employee,fact_record, on = "Employee_ID" , how = 'inner')
employee_retention_by_dept =  merge_df.pivot_table(index = 'Department', columns = 'Resigned', values='Employee_ID',aggfunc='count').fillna(0)
employee_retention_by_dept['Percent_Resign'] = (employee_retention_by_dept[True]*100)/(employee_retention_by_dept[True]+employee_retention_by_dept[False])
employee_retention_by_dept

Resigned,False,True,Percent_Resign
Department,,,
Customer Support,8659,6115,41.390280
Engineering,15389,4620,23.089610
Finance,4335,670,13.386613
HR,3470,557,13.831636
IT,15340,4556,22.899075
Legal,1757,252,12.543554
Marketing,2306,1679,42.132999
Operations,11585,3401,22.694515
Sales,8942,6367,41.589914


#### 1.10 Remote work frequency and department wise, performance score relation.

In [3]:
merge_df_1 = pd.merge(fact_workload,fact_record, on = "Employee_ID" , how = 'inner')
merge_df_full = pd.merge(merge_df_1,dim_employee, on = "Employee_ID" , how = 'inner')
remote_work_perf_score = merge_df_full.pivot_table(index = 'Remote_Work_Frequency', columns = 'Department', values = 'Performance_Score',aggfunc = 'mean').fillna(0)
remote_work_perf_score.round(2)

Department,Customer Support,Engineering,Finance,HR,IT,Legal,Marketing,Operations,Sales
Remote_Work_Frequency,,,,,,,,,
0,3.20,3.21,3.20,3.19,3.19,3.24,3.22,3.21,3.19
20,3.19,3.19,3.22,3.22,3.19,3.22,3.19,3.19,3.17
50,3.17,3.19,3.21,3.20,3.16,3.16,3.19,3.20,3.19
80,3.16,3.18,3.24,3.19,3.21,3.17,3.14,3.21,3.21
100,3.21,3.23,3.18,3.31,3.19,3.13,3.22,3.23,3.22
